# QC: ALS/FTD

**Start date:** 10-10-2023

**End date:** 2024-11-20

**Analysed by:** Ruth Chia

**Working directory:** `/data/ALS_50k/OLINK/Results/Oct2023`

___


## Prep OLINK UKBB dataset

In [ ]:
import pandas as pd
import numpy as np
ukb = pd.read_csv("UKB/2024-09-17_olink_instance_0.csv")
ukb.head(2)

In [ ]:
pheno = pd.read_csv("/data/NDRS_LNG/UKBB/extract_UKBB_codes_May2022/COVARIATES.UKBB_EURO_ALS_controls_unrelated.txt",sep="\t")
print(pheno.columns)
pheno.head(2)

In [ ]:
olink_pheno_als_ctrl = pheno.merge(ukb, how="inner", left_on='IID', right_on='eid')
olink_pheno_als_ctrl.to_csv("UKB/PHENO_with_OLINK_UKB_256ALS_37029NonNeuroControls.txt",sep="\t", index=False, header=True)
print(olink_pheno_als_ctrl.shape)
print(olink_pheno_als_ctrl.ALS_AFFECTION_STATUS.value_counts())

In [ ]:
ukb_others = ukb[~(ukb.eid.isin(olink_pheno_als_ctrl.IID))]
ukb_others.to_csv("UKB/OLINK_UKB_15729othersamples_otherindications.txt",sep="\t", index=False, header=True)
ukb_others.shape

## Pre-analysis QC stats check

### Heparin

In [ ]:
%%bash
module load R/4.3

require(data.table)
require(tidyverse)
library(readxl)

# read in results, manifest and well_index_key files
results_heparin <- fread("2023_04_25_Scholz_NPX_2023-08-30/2023_04_25_Scholz_NPX_2023-08-30.csv",sep=";")
results_edta <- fread("2023_04_25_Traynor_NPX_2023-09-27/2023_04_25_Traynor_NPX_2023-09-27.csv",sep=";")
manifest_heparin <- read_excel("Olink_project_final_10.12.2023_updatedRuth.xlsx", sheet = "Heparin_selected") %>% data.frame()
manifest_edta <- read_excel("Olink_project_final_10.12.2023_updatedRuth.xlsx", sheet = "EDTA_selected") %>% data.frame()

index <- fread("../July2023/Well_index_map.txt")

# add new column in results df to indicate corresponding Olink_Plate_No
results_heparin$Olink_Plate_No <- results_heparin$PlateID
results_heparin$Olink_Plate_No <- gsub("Explore_SS_Heparin_Plate5","TRAYNOR_5", results_heparin$Olink_Plate_No)
results_heparin$Olink_Plate_No <- gsub("Plate_6_Heparin_CSV","TRAYNOR_6", results_heparin$Olink_Plate_No)
results_heparin$Olink_Plate_No <- gsub("Plate_7_Heparin_CSV","TRAYNOR_7", results_heparin$Olink_Plate_No)
results_heparin$Olink_Plate_No <- gsub("Plate_8_Heparin_CSV","TRAYNOR_8", results_heparin$Olink_Plate_No)
results_heparin$Olink_Plate_No <- gsub("Plate9_Hep_CSV","TRAYNOR_9", results_heparin$Olink_Plate_No)
table(results_heparin$Olink_Plate_No)

results_edta$Olink_Plate_No <- results_edta$PlateID
results_edta$Olink_Plate_No <- gsub("Explore_Plate_Layout_BT_EDTA_Plate1_CSV","NDRU_1", results_edta$Olink_Plate_No)
results_edta$Olink_Plate_No <- gsub("Plate2_EDTA_CSV","NDRU_2", results_edta$Olink_Plate_No)
results_edta$Olink_Plate_No <- gsub("Plate3_EDTA_CSV","NDRU_3", results_edta$Olink_Plate_No)
results_edta$Olink_Plate_No <- gsub("Plate4_EDTA_CSV","NDRU_4", results_edta$Olink_Plate_No)
table(results_edta$Olink_Plate_No)

dim(results_edta)
dim(results_heparin)
dim(manifest_edta)
dim(manifest_heparin)
dim(index)

head(results_edta)
head(results_heparin)
head(manifest_edta)
head(manifest_heparin)
head(index)

table(manifest_heparin$Olink_Plate_Position, manifest_heparin$Olink_Plate_No)

table(manifest_edta$Olink_Plate_Position, manifest_edta$Olink_Plate_No)

# merge manifest with index df so that it has a corresponding Index number --> use this then to merge with results file
manifest_w_index_heparin <- merge(manifest_heparin, index, by.x="Olink_Plate_Position", by.y="Well", all.x=T)
dim(manifest_w_index_heparin)
head(manifest_w_index_heparin)

# merge manifest with index df so that it has a corresponding Index number --> use this then to merge with results file
manifest_w_index_edta <- merge(manifest_edta, index, by.x="Olink_Plate_Position", by.y="Well", all.x=T)
dim(manifest_w_index_edta)
head(manifest_w_index_edta)

# merge updated manifest w index info with results --> then check to make sure sample ID are the same
merge_heparin <- merge(results_heparin, manifest_w_index_heparin, by=c("Olink_Plate_No","Index"), all.x=T)
dim(merge_heparin)

# edit diagnosis column for SC and BSCP samples to indicate what these samples are
temp_1 <- merge_heparin[grepl("SC_", merge_heparin$SampleID),] %>% mutate(Diagnosis = "NIA sample control")
temp_2 <- merge_heparin[grepl("BSCP_", merge_heparin$SampleID),] %>% mutate(Diagnosis = "Pooled bridging sample")
temp_3 <- rbind(temp_1,temp_2)
temp_4 <- subset(merge_heparin, !(merge_heparin$SampleID %in% temp_3$SampleID))
results_manifest_heparin <- rbind(temp_3, temp_4)
results_manifest_heparin$QC_Warning <- factor(results_manifest_heparin$QC_Warning, levels=c("PASS","WARN"))

dim(results_manifest_heparin)
colnames(results_manifest_heparin)
head(results_manifest_heparin)

results_manifest_heparin$SampleMatch <- ifelse(results_manifest_heparin$SampleID == results_manifest_heparin$`ID.cryovial...21`, "yes", "no")
results_manifest_heparin <- results_manifest_heparin %>% mutate_if(is.character, str_replace_all, ' ', '_')
table(results_manifest_heparin$SampleMatch) #checked results file and manifest file - all samples in corresponding positions are a match! VERY GOOD!

# save out final/compiled results file 
write.table(results_manifest_heparin, "OLINK_results_compiled_HEPARIN.txt", sep="\t", quote=F, row.names=F, col.names=T)

table(results_manifest_heparin$Index, results_manifest_heparin$Olink_Plate_No)

#### make some NPX distribution density plots by sample and check range

In [ ]:
%%bash
module load R/4.3
R --vanilla --no-save

require(data.table)
require(tidyverse)

results_manifest_heparin <- fread("OLINK_results_compiled_HEPARIN.txt")

library(plotly)
options(repr.plot.width=24, repr.plot.height=12)

NPX_density_plot <- ggplot(data = results_manifest_heparin, aes(x=NPX, color=SampleID)) +
                    geom_density(alpha=0.1) +
                    facet_grid(Panel ~ .) +
                    guides(color = F) +
                    theme_bw()

p <- ggplotly(NPX_density_plot)
p

results_manifest_heparin %>%
group_by(Panel,Olink_Plate_No) %>%
summarize(mean_NPX = mean(NPX, na.rm = TRUE),
          sd_NPX = sd(NPX, na.rm = TRUE),
          iqr_NPX = IQR(NPX, na.rm = TRUE),
          mad_NPX = mad(NPX, na.rm = TRUE))

summ_stats <- results_manifest_heparin %>%
              group_by(SampleID) %>%
              summarize(mean_NPX = mean(NPX, na.rm = TRUE),
              sd_NPX = sd(NPX, na.rm = TRUE),
              lower_iqr_NPX = quantile(NPX, na.rm = TRUE)[2],
              upper_iqr_NPX = quantile(NPX, na.rm = TRUE)[4],
              iqr_NPX = IQR(NPX, na.rm = TRUE),
              mad_NPX = mad(NPX, na.rm = TRUE)) %>%
              data.frame()
summ_stats          

# flag samples with IQR beyond the mean to see which samples have extreme values
min_threshold <- -0.3802 - 1.5*(-0.2106 - (-0.4979))
max_threshold <- 0.3692 + 1.5*(0.4985 - (0.1842))
outliers <- summ_stats %>% filter(lower_iqr_NPX < min_threshold  | upper_iqr_NPX > max_threshold)
dim(outliers)
outliers %>% arrange(SampleID)

results_manifest_heparin$possible_outliers <- ifelse(results_manifest_heparin$SampleID %in% outliers$SampleID, "yes", "no")
# save out merged + updated results/manifest file for future loadings
write.table(results_manifest_heparin, "Updated-merged_results_manifest_heparin_2024-03-12.txt", sep="\t", quote=F,row.names=F,col.names=T)

table(results_manifest_heparin$Diagnosis, results_manifest_heparin$possible_outliers)

breakdown <- results_manifest_heparin %>% filter(possible_outliers == "yes") %>% select(SampleID, Diagnosis, Sex) %>% distinct()
table(breakdown$Diagnosis, breakdown$Sex)

### EDTA

In [ ]:
%%bash
module load R/4.3

require(data.table)
require(tidyverse)
library(readxl)

# read in results, manifest and well_index_key files
results_heparin <- fread("2023_04_25_Scholz_NPX_2023-08-30/2023_04_25_Scholz_NPX_2023-08-30.csv",sep=";")
results_edta <- fread("2023_04_25_Traynor_NPX_2023-09-27/2023_04_25_Traynor_NPX_2023-09-27.csv",sep=";")
manifest_heparin <- read_excel("Olink_project_final_10.12.2023_updatedRuth.xlsx", sheet = "Heparin_selected") %>% data.frame()
manifest_edta <- read_excel("Olink_project_final_10.12.2023_updatedRuth.xlsx", sheet = "EDTA_selected") %>% data.frame()

index <- fread("../July2023/Well_index_map.txt")

# add new column in results df to indicate corresponding Olink_Plate_No
results_heparin$Olink_Plate_No <- results_heparin$PlateID
results_heparin$Olink_Plate_No <- gsub("Explore_SS_Heparin_Plate5","TRAYNOR_5", results_heparin$Olink_Plate_No)
results_heparin$Olink_Plate_No <- gsub("Plate_6_Heparin_CSV","TRAYNOR_6", results_heparin$Olink_Plate_No)
results_heparin$Olink_Plate_No <- gsub("Plate_7_Heparin_CSV","TRAYNOR_7", results_heparin$Olink_Plate_No)
results_heparin$Olink_Plate_No <- gsub("Plate_8_Heparin_CSV","TRAYNOR_8", results_heparin$Olink_Plate_No)
results_heparin$Olink_Plate_No <- gsub("Plate9_Hep_CSV","TRAYNOR_9", results_heparin$Olink_Plate_No)
table(results_heparin$Olink_Plate_No)

results_edta$Olink_Plate_No <- results_edta$PlateID
results_edta$Olink_Plate_No <- gsub("Explore_Plate_Layout_BT_EDTA_Plate1_CSV","NDRU_1", results_edta$Olink_Plate_No)
results_edta$Olink_Plate_No <- gsub("Plate2_EDTA_CSV","NDRU_2", results_edta$Olink_Plate_No)
results_edta$Olink_Plate_No <- gsub("Plate3_EDTA_CSV","NDRU_3", results_edta$Olink_Plate_No)
results_edta$Olink_Plate_No <- gsub("Plate4_EDTA_CSV","NDRU_4", results_edta$Olink_Plate_No)
table(results_edta$Olink_Plate_No)

dim(results_edta)
dim(results_heparin)
dim(manifest_edta)
dim(manifest_heparin)
dim(index)

head(results_edta)
head(results_heparin)
head(manifest_edta)
head(manifest_heparin)
head(index)

table(manifest_heparin$Olink_Plate_Position, manifest_heparin$Olink_Plate_No)

table(manifest_edta$Olink_Plate_Position, manifest_edta$Olink_Plate_No)

# merge manifest with index df so that it has a corresponding Index number --> use this then to merge with results file
manifest_w_index_heparin <- merge(manifest_heparin, index, by.x="Olink_Plate_Position", by.y="Well", all.x=T)
dim(manifest_w_index_heparin)
head(manifest_w_index_heparin)

# merge manifest with index df so that it has a corresponding Index number --> use this then to merge with results file
manifest_w_index_edta <- merge(manifest_edta, index, by.x="Olink_Plate_Position", by.y="Well", all.x=T)
dim(manifest_w_index_edta)
head(manifest_w_index_edta)

# merge updated manifest w index info with results --> then check to make sure sample ID are the same
merge_edta <- merge(results_edta, manifest_w_index_edta, by=c("Olink_Plate_No","Index"), all.x=T) %>% filter(Olink_Plate_No != "Extras")
dim(merge_edta)

# edit diagnosis column for SC and BSCP samples to indicate what these samples are
temp_1 <- merge_edta[grepl("SC_", merge_edta$SampleID),] %>% mutate(Diagnosis = "NIA sample control")
temp_2 <- merge_edta[grepl("BSCP_", merge_edta$SampleID),] %>% mutate(Diagnosis = "Pooled bringing sample")
temp_3 <- rbind(temp_1,temp_2)
temp_4 <- subset(merge_edta, !(merge_edta$SampleID %in% temp_3$SampleID))
results_manifest_edta <- rbind(temp_3, temp_4)
results_manifest_edta$QC_Warning <- factor(results_manifest_edta$QC_Warning, levels=c("PASS","WARN"))

dim(results_manifest_edta)
colnames(results_manifest_edta)
head(results_manifest_edta)

results_manifest_edta$SampleMatch <- ifelse(results_manifest_edta$SampleID == results_manifest_edta$Sample_ID, "yes", "no")
results_manifest_edta <- results_manifest_edta %>% mutate_if(is.character, str_replace_all, ' ', '_')
table(results_manifest_edta$SampleMatch) # No sample mismatched! VERY good - any mismatches were corrected :)

# save out final/compiled results file 
write.table(results_manifest_edta, "OLINK_results_compiled_EDTA.txt", sep="\t", quote=F, row.names=F, col.names=T)
table(results_manifest_edta$Index, results_manifest_edta$Olink_Plate_No)
table(results_manifest_edta$Diagnosis)

results_manifest_edta %>%
group_by(Panel,Olink_Plate_No) %>%
summarize(mean_NPX = mean(NPX, na.rm = TRUE),
          sd_NPX = sd(NPX, na.rm = TRUE),
          iqr_NPX = IQR(NPX, na.rm = TRUE),
          mad_NPX = mad(NPX, na.rm = TRUE))

summ_stats <- results_manifest_edta %>%
              group_by(SampleID) %>%
              summarize(mean_NPX = mean(NPX, na.rm = TRUE),
              sd_NPX = sd(NPX, na.rm = TRUE),
              lower_iqr_NPX = quantile(NPX, na.rm = TRUE)[2],
              upper_iqr_NPX = quantile(NPX, na.rm = TRUE)[4],
              iqr_NPX = IQR(NPX, na.rm = TRUE),
              mad_NPX = mad(NPX, na.rm = TRUE)) %>%
              data.frame()

summ_stats
summary(summ_stats)

# flag samples with IQR beyond the mean to see which samples have extreme values
min_threshold <- -0.3457 - 1.5*(-0.1813 - (-0.4711))
max_threshold <- 0.4268 + 1.5*(0.6222 - (0.1708))
outliers <- summ_stats %>% filter(lower_iqr_NPX < min_threshold  | upper_iqr_NPX > max_threshold)
dim(outliers)
outliers %>% arrange(SampleID)

results_manifest_edta$possible_outliers <- ifelse(results_manifest_edta$SampleID %in% outliers$SampleID, "yes", "no")
# save out merged + updated results/manifest file for future loadings
write.table(results_manifest_edta, "Updated-merged_results_manifest_edta_2024-03-12.txt", sep="\t", quote=F,row.names=F,col.names=T)

table(results_manifest_edta$Diagnosis, results_manifest_edta$possible_outliers)

breakdown <- results_manifest_edta %>% filter(possible_outliers == "yes") %>% select(SampleID, Diagnosis, Sex) %>% distinct()
table(breakdown$Diagnosis, breakdown$Sex)

## Generate genetic PCs for samples

### Heparin

Note: most the the samples were sequenced as part of DementiaSeq1 

Path to QC-ed WGS data: `/data/ALS_50k/DementiaSeq.TopmedJointCalled.June2020//FILTERED.freeze9.LBD.FTD.ALLcontrols4K.chr*.pgen/pavr/psam`

In [ ]:
!mkdir PCA_Heparin

In [ ]:
%%bash
cd PCA_Heparin/
module load R/4.3
R --vanilla --no-save

# load libraries
require(data.table)
require(tidyverse)
library("Hmisc")
library(ggplot2) 
library(ggpubr)
library(ggrepel)
library(broom)
library(OlinkAnalyze)

# read in results files
data <- fread("../OLINK_results_compiled_HEPARIN.txt")
all_samples_data <- data %>%
                    select(SampleID,Genome_ID,Diagnosis) %>%
                    distinct()
table(all_samples_data$Diagnosis)

# make list of samples to generate PCs for - note that the column to use is the Genome_ID column
samples <- data %>%
           select(SampleID,Genome_ID,Diagnosis) %>%
           distinct() %>%
           filter(Diagnosis == "Control" | Diagnosis == "ALS") %>%
           filter(Genome_ID != "NA") %>%
           data.frame()
samples_list <- samples %>% select(Genome_ID) %>% distinct() %>% filter(Genome_ID != "NA") %>%
                rename(FID = Genome_ID) %>% mutate(IID = FID)
write.table(samples, "SampleInfo.Heparin_forWGSextraction.txt",sep="\t",quote=F,row.names=F,col.names=T)
write.table(samples_list, "SampleList.Heparin_forWGSextraction.txt",sep="\t",quote=F,row.names=F,col.names=T)

temp <- samples[,c("Genome_ID","Diagnosis")] %>% distinct()
table(temp$Diagnosis)

list_with_dup_RES <- samples %>% group_by(Genome_ID) %>% count() %>% filter(n>1 & Genome_ID != "NA")
subset(data, data$Genome_ID %in% list_with_dup_RES$Genome_ID) %>% select(Olink_Plate_No,SampleID,Diagnosis,Genome_ID) %>% distinct() %>% arrange(Genome_ID)

In [ ]:
%%bash
cd PCA_Heparin/

#module load plink/1.9.0-beta4.4
#module load R/4.3
#module load GCTA/1.94.1
module load flashpca/2.0
module load plink/3.6-alpha

DATA="/data/ALS_50k/DementiaSeq.TopmedJointCalled.June2020"

# Prune snps --> merge to a single file
for CHNUM in {1..22};
do
plink2 \
--pfile $DATA/FTD/CLEAN.UNRELATED/FILTERED.FTD.controls.UNRELATED_chr${CHNUM} \
--extract $DATA/SNV.Variant.List/preQC.SNV.Varlist.neurod.freeze9.LBD.FTD.ALLcontrols4K.chr${CHNUM}.txt \
--keep SampleList.Heparin_forWGSextraction.txt \
--remove-nosex \
--maf 0.01 \
--geno 0.01 \
--hwe 5e-6 \
--autosome \
--exclude range /data/ALS_50k/DementiaSeq.TopmedJointCalled.June2020/FTD/Analysis.GLM/GLM.FTD.4Kcontrols/scripts/GenomicRangeExlusion.forPCA.txt \
--make-bed \
--out temp_chr${CHNUM} \
--memory 119500 --threads 19;

plink2 \
--bfile temp_chr${CHNUM} \
--remove-nosex \
--geno 0.01 \
--maf 0.05 \
--indep-pairwise 1000 10 0.02 \
--out temp_pruning_chr${CHNUM} \
--memory 119500 --threads 19;

plink2 \
--bfile temp_chr${CHNUM} \
--remove-nosex \
--extract temp_pruning_chr${CHNUM}.prune.in \
--keep-allele-order \
--make-bed \
--out pruned_chr${CHNUM} \
--memory 119500 --threads 19;

done

# Merge pruned binaries to single file with chr 1-22
cat pruned_chr1.fam > pruned_ALLchr_forPCA.fam
for chr in {1..22}; do cat pruned_chr${chr}.bim; done > pruned_ALLchr_forPCA.bim
(echo -en "\x6C\x1B\x01"; for chr in {1..22}; do tail -c +4 pruned_chr${chr}.bed; done) > pruned_ALLchr_forPCA.bed

# Calculate/generate PCs based on pruned data set
flashpca --bfile pruned_ALLchr_forPCA --suffix _FILTERED.FTD.controls_ALLchr_forPCA --numthreads 19

# Move all log files to a new folder
mkdir logFiles
mv *.log logFiles

# Remove intermediate files
rm temp_*
rm pruned*_chr*.*


In [ ]:
%%bash
cd PCA_Heparin/
module load R/4.3
R --vanilla --no-save

# load libraries
require(data.table)
require(tidyverse)
library("Hmisc")
library(ggplot2) 
library(ggpubr)
library(ggrepel)
library(broom)
library(OlinkAnalyze)

# read in results files and pcs file
data <- fread("../OLINK_results_compiled_HEPARIN.txt")
data$Age_Consensus <- ifelse(!is.na(data$Age_Onset), data$Age_Onset, data$Age_Collection)
pcs <- fread("pcs_FILTERED.FTD.controls_ALLchr_forPCA",sep="\t") %>% select(-FID) %>% rename(Genome_ID = IID)

dim(data)
dim(pcs)

# merge data results and pcs
cov <- merge(data, pcs, by="Genome_ID", all.x=T)
write.table(cov, "../OLINK_results_compiled_HEPARIN_wPCs.txt", sep="\t", quote=F, row.names=F, col.names=T)
                        
# run step() to determine which PCs to correct for for the case/control status
temp <- cov %>%
        filter(Diagnosis == "ALS" | Diagnosis == "Control") %>%
        select(SampleID,Diagnosis,Sex,Age_Onset,Age_Consensus,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10) %>%
        distinct()
temp$CASE <- ifelse(temp$Diagnosis == "ALS", 1, 0)
temp$GENDER <- ifelse(temp$Sex == "Female", 1, 0)
table(temp$Diagnosis, temp$GENDER)
table(temp$CASE, temp$Sex)

# model using glm() and then use step() to determine the model that has the lowest AIC
# this tell us which covariates you included in the model are important to use/for adjustment in your analysis
model <- glm(CASE ~ GENDER + Age_Consensus + PC1 + PC2 + PC3 + PC4 + PC5 + PC6 + PC7 + PC8 + PC9 + PC10 , data = temp, family = "binomial"(link = "logit"))
summary(model)
step(model)

# run step() to determine which PCs to correct for for the AAO for cases only
model <- glm(Age_Onset ~ GENDER + PC1 + PC2 + PC3 + PC4 + PC5 + PC6 + PC7 + PC8 + PC9 + PC10 , data = temp %>% filter(Diagnosis=="ALS"), family = "gaussian")
summary(model)
step(model)


**Genetic PCs to adjust for in the glm for the ALS/control heparin cohort:** `PC3,PC4`

**Genetic PCs to adjust for in the glm for the AAO-ALS heparin cohort:** `PC1,PC2`

### EDTA

The samples in the EDTA plates are a little mixed i.e. some samples were part of DementiaSeq1, DementiaSeq2, and some were only chip-genotyped.

What I will need to do here is to identify which samples were WGS-ed on which project and which were chip genotyped. Then, will need to subset those samples, make a list of overlapping variants shared across all subsets, and then force merge the cohorts, prune snps and generated genetic PCs. 

Will also need to update the phenotype file to include the generated PCs, and also include one additional column to indicate where the genetic data/PCs were came from. 

Paths to genetic data that I can still digging into:
1. DementiaSeq1: `/data/ALS_50k/DementiaSeq_QC/FILTERED.neurod.june2019.exclTOPMED.noDuplicates.noSpanningDels.hwe1e-6.hg38.chrX.psam`
2. DementiaSeq2: `/data/ALS_50k/BrainBankSeq_May2021/LBD.FTD/subset.LBD_FTD_RESv2/CLEAN.LBD.FTD.controls.res2.may2021.noSpanningDels.chr2.psam`
3. MSA: `/data/ALS_50k/BrainBankSeq_May2021/MSA/WGS/MSA.keepRelated/FILTERED.MSA.keepRelated.controls_chr1.psam`
4. Chip-genotyping: `/data/ALS_50k/OLINK/Results/Oct2023/Neurochip_olink/plink_output/NDRU_olink_genotyping.ped`

In [ ]:
!wc -l /data/ALS_50k/DementiaSeq_QC/FILTERED.neurod.june2019.exclTOPMED.noDuplicates.noSpanningDels.hwe1e-6.hg38.chrX.psam
!wc -l /data/ALS_50k/BrainBankSeq_May2021/LBD.FTD/subset.LBD_FTD_RESv2/CLEAN.LBD.FTD.controls.res2.may2021.noSpanningDels.chr2.psam
!wc -l /data/ALS_50k/BrainBankSeq_May2021/MSA/WGS/MSA.keepRelated/FILTERED.MSA.keepRelated.controls_chr1.psam
!wc -l /data/ALS_50k/OLINK/Results/Oct2023/Neurochip_olink/plink_output/NDRU_olink_genotyping.ped

In [ ]:
!grep "RES08007" /data/ALS_50k/DementiaSeq_QC/GRM/FILTERED.to.remove.GRM_matrix_duplicates.grm.SampleINFO.txt

In [ ]:
!mkdir PCA_EDTA

In [ ]:
%%bash
cd Neurochip_olink/plink_output/

# convert .ped/.map plink files to plink1 format
module load plink/1.9.0-beta4.4

plink \
--file NDRU_olink_genotyping \
--make-bed \
--out NDRU_olink_genotyping

In [ ]:
%%bash
cd PCA_EDTA/
module load R/4.3
R --vanilla --no-save

# load libraries
require(data.table)
require(tidyverse)

# read in OLINK results file; get list of samples in the EDTA cohort
data <- fread("../OLINK_results_compiled_EDTA.txt")

# note that for one BLSA sample (RES08007) - this is a duplicate sample with another sample i.e. RES02508
# update this info in the file so that I cann match this back to the genomic data
data$Genome_ID <- gsub("RES08007", "RES02508", data$Genome_ID)
write.table(data, "../OLINK_results_compiled_EDTA_updated.txt", sep="\t", quote=F, row.names=F, col.names=T)

# make list of samples to exclude
to_exclude <- c("SC_1","SC_2","LNG_NDRU131185_EDTA")
all_samples_data <- subset(data, !(data$SampleID %in% to_exclude)) %>%
                    filter(Diagnosis != "Pooled_bringing_sample") %>%
                    select(SampleID,Genome_ID,NeuroChip_ID,Diagnosis, Sample_ID, Genome_ID) %>%
                    distinct()
all_samples_data[is.na(all_samples_data)] <- "NA"
all_samples_data$WGS_data <- ifelse(all_samples_data$Genome_ID == "NA", "no","yes")
all_samples_data$Neurochip_data <- ifelse(all_samples_data$NeuroChip_ID == "NA", "no","yes")
all_samples_data$Genomic_data <- ifelse(all_samples_data$WGS_data == "yes" | all_samples_data$Neurochip_data == "yes", "yes", "no")

table(all_samples_data$Diagnosis)
table(all_samples_data$Diagnosis,all_samples_data$WGS_data)
table(all_samples_data$Diagnosis,all_samples_data$WGS_data,all_samples_data$Neurochip_data)
table(all_samples_data$Diagnosis,all_samples_data$Genomic_data)
# good news is that most of the samples have genomic data (either WGS/chip-genotyped) - 11 samples do not have genetic data



# now need to check where the genomic data is for each sample
ds1 <- fread("/data/ALS_50k/DementiaSeq_QC/FILTERED.neurod.june2019.exclTOPMED.noDuplicates.noSpanningDels.hwe1e-6.hg38.chrX.psam") %>% select(IID)
ds2 <- fread("/data/ALS_50k/BrainBankSeq_May2021/LBD.FTD/subset.LBD_FTD_RESv2/CLEAN.LBD.FTD.controls.res2.may2021.noSpanningDels.chr2.psam") %>% select(IID)
msa <- fread("/data/ALS_50k/BrainBankSeq_May2021/MSA/WGS/MSA.keepRelated/FILTERED.MSA.keepRelated.controls_chr1.psam") %>% select(IID)
chip <- fread("/data/ALS_50k/OLINK/Results/Oct2023/Neurochip_olink/plink_output/NDRU_olink_genotyping.fam", header=F) %>% select(V2) %>% rename(IID = V2)

olink_ds1 <- subset(all_samples_data, all_samples_data$Genome_ID %in% ds1$IID)  %>% mutate(Genetic_data = "DementiaSeq_1")
temp1 <- subset(all_samples_data, !(all_samples_data$SampleID %in% olink_ds1$SampleID))
olink_ds2 <- subset(temp1, temp1$Genome_ID %in% ds2$IID) %>% mutate(Genetic_data = "DementiaSeq_2")
temp2 <- subset(temp1, !(temp1$SampleID %in% olink_ds2$SampleID))
olink_msa <- subset(temp2, temp2$Genome_ID %in% msa$IID) %>% mutate(Genetic_data = "BBSeq_MSA")
temp3 <- subset(temp2, !(temp2$SampleID %in% olink_msa$SampleID))
olink_chip <- subset(temp3, temp3$SampleID %in% chip$IID) %>% mutate(Genetic_data = "NDRU-chip-genotyped")

dim(olink_ds1)
dim(olink_ds2)
dim(olink_msa)
dim(olink_chip)

# make sample lists for plink subsets
sampleList_olink_ds1 <- olink_ds1 %>% select(Genome_ID) %>% rename(FID = Genome_ID) %>% mutate(IID = FID)
sampleList_olink_ds2 <- olink_ds2 %>% select(Genome_ID) %>% rename(FID = Genome_ID) %>% mutate(IID = FID)
sampleList_olink_msa <- olink_msa %>% select(Genome_ID) %>% rename(FID = Genome_ID) %>% mutate(IID = FID)
sampleList_olink_chip <- olink_chip %>% select(SampleID) %>% rename(FID = SampleID) %>% mutate(IID = FID)
write.table(sampleList_olink_ds1, "SampleList.OLINK_DementiaSeq1.FID_IID.txt",sep="\t", quote=F, row.names=F, col.names=T)
write.table(sampleList_olink_ds2, "SampleList.OLINK_DementiaSeq2.FID_IID.txt",sep="\t", quote=F, row.names=F, col.names=T)
write.table(sampleList_olink_msa, "SampleList.OLINK_BBSeq_MSA.FID_IID.txt",sep="\t", quote=F, row.names=F, col.names=T)
write.table(sampleList_olink_chip, "SampleList.OLINK_NDRU-chip-genotyped.FID_IID.txt",sep="\t", quote=F, row.names=F, col.names=T)

# create dataframe for samples with genetic data
olink_w_geneticdata <- rbind(olink_ds1,olink_ds2,olink_msa,olink_chip)
dim(olink_w_geneticdata)
# merge genetic info path to original data file and save it
data_updated <- merge(data, olink_w_geneticdata[,c("SampleID","Genetic_data")], by="SampleID", all.x=T)
write.table(olink_w_geneticdata, "../OLINK_results_compiled_EDTA_updated_w-genetic-info.txt", sep="\t", quote=F, row.names=F, col.names=T)

# check which samples have no corresponding genetic data
missing_geneticdata <- subset(all_samples_data, !(all_samples_data$SampleID %in% olink_w_geneticdata$SampleID))
missing_geneticdata
## only the 11 that was originally flagged as not having genetic data is matched!

#### Subset genetic data

QC-ed plink file paths to subset from:
1. DementiaSeq1: `/data/ALS_50k/DementiaSeq_QC/FILTERED.neurod.june2019.exclTOPMED.noDuplicates.noSpanningDels.hwe1e-6.VariantFreqOutliersRemoved.hg38.chr*.pgen/pvar/psam`
2. DementiaSeq2: `/data/ALS_50k/BrainBankSeq_May2021/LBD.FTD/subset.LBD_FTD_RESv2/CLEAN.LBD.FTD.controls.res2.may2021.noSpanningDels.chr*.pgen/pvar/psam`
3. MSA: `/data/ALS_50k/BrainBankSeq_May2021/MSA/WGS/MSA.keepRelated/FILTERED.MSA.keepRelated.controls_chr*.pgen/pvar/psam`
4. Chip-genotyping: `/data/ALS_50k/OLINK/Results/Oct2023/Neurochip_olink/plink_output/FILTERED.NDRU.OLINK.hg19.keepRelated.bed/bim/fam`

Sample list to extract from each genetic cohort:
1. DementiaSeq1: `/data/ALS_50k/OLINK/Results/Oct2023/PCA_EDTA/SampleList.OLINK_DementiaSeq1.FID_IID.txt`
2. DementiaSeq2: `/data/ALS_50k/OLINK/Results/Oct2023/PCA_EDTA/SampleList.OLINK_DementiaSeq2.FID_IID.txt`
3. MSA: `/data/ALS_50k/OLINK/Results/Oct2023/PCA_EDTA/SampleList.OLINK_BBSeq_MSA.FID_IID.txt`
4. Chip-genotyping: `/data/ALS_50k/OLINK/Results/Oct2023/PCA_EDTA/SampleList.OLINK_NDRU-chip-genotyped.FID_IID.txt`


In [ ]:
!mkdir PCA_EDTA/subset_genetic_data
!mkdir PCA_EDTA/subset_genetic_data/DementiaSeq1
!mkdir PCA_EDTA/subset_genetic_data/DementiaSeq2
!mkdir PCA_EDTA/subset_genetic_data/BBSeq_MSA

In [ ]:
!mkdir PCA_EDTA/subset_genetic_data/NDRU-chip-genotyped

##### Make list of common variants across genetic subsets
will use the NDRU-chip dataset as reference as this has smaller number of variants

In [ ]:
%%bash
cd PCA_EDTA/subset_genetic_data

module load plink/3.6-alpha

# NDRU-chip-genotyped
plink2 \
--bfile /data/ALS_50k/OLINK/Results/Oct2023/Neurochip_olink/plink_output/FILTERED.NDRU.OLINK.hg38.keepRelated.autosomes \
--autosome \
--rm-dup \
--keep /data/ALS_50k/OLINK/Results/Oct2023/PCA_EDTA/SampleList.OLINK_NDRU-chip-genotyped.FID_IID.txt \
--make-bed \
--out NDRU-chip-genotyped/temp

plink2 \
--bfile /data/ALS_50k/OLINK/Results/Oct2023/Neurochip_olink/plink_output/FILTERED.NDRU.OLINK.hg38.keepRelated.autosomes \
--exclude NDRU-chip-genotyped/temp.rmdup.mismatch \
--make-bed \
--out NDRU-chip-genotyped/OLINK_NDRU-chip-genotyped

plink2 \
--bfile NDRU-chip-genotyped/OLINK_NDRU-chip-genotyped \
--write-snplist allow-dups \
--out NDRU-chip-genotyped/VarList.OLINK_NDRU-chip-genotyped

rm NDRU-chip-genotyped/temp*

sort NDRU-chip-genotyped/VarList.OLINK_NDRU-chip-genotyped.snplist | uniq > NDRU-chip-genotyped/VarList.OLINK_NDRU-chip-genotyped.nodups.snplist
wc -l NDRU-chip-genotyped/VarList.OLINK_NDRU-chip-genotyped.snplist
wc -l NDRU-chip-genotyped/VarList.OLINK_NDRU-chip-genotyped.nodups.snplist

In [ ]:
%%bash
cd PCA_EDTA/subset_genetic_data

module load plink/3.6-alpha

# DementiaSeq1
for CHNUM in {1..22}
do
plink2 \
--pfile /data/ALS_50k/DementiaSeq_QC/FILTERED.neurod.june2019.exclTOPMED.noDuplicates.noSpanningDels.hwe1e-6.VariantFreqOutliersRemoved.hg38.chr${CHNUM} \
--keep /data/ALS_50k/OLINK/Results/Oct2023/PCA_EDTA/SampleList.OLINK_DementiaSeq1.FID_IID.txt \
--extract NDRU-chip-genotyped/VarList.OLINK_NDRU-chip-genotyped.nodups.snplist \
--write-snplist \
--out DementiaSeq1/VarList.OLINK_DementiaSeq1_chr${CHNUM}
done

# DementiaSeq2
for CHNUM in {1..22}
do
plink2 \
--pfile /data/ALS_50k/BrainBankSeq_May2021/LBD.FTD/subset.LBD_FTD_RESv2/CLEAN.LBD.FTD.controls.res2.may2021.noSpanningDels.chr${CHNUM} \
--keep /data/ALS_50k/OLINK/Results/Oct2023/PCA_EDTA/SampleList.OLINK_DementiaSeq2.FID_IID.txt \
--extract NDRU-chip-genotyped/VarList.OLINK_NDRU-chip-genotyped.nodups.snplist \
--write-snplist \
--out DementiaSeq2/VarList.OLINK_DementiaSeq2_chr${CHNUM}
done

# BBSeq_MSA
for CHNUM in {1..22}
do
plink2 \
--pfile /data/ALS_50k/BrainBankSeq_May2021/MSA/WGS/MSA.keepRelated/FILTERED.MSA.keepRelated.controls_chr${CHNUM} \
--keep /data/ALS_50k/OLINK/Results/Oct2023/PCA_EDTA/SampleList.OLINK_BBSeq_MSA.FID_IID.txt \
--extract NDRU-chip-genotyped/VarList.OLINK_NDRU-chip-genotyped.nodups.snplist \
--write-snplist \
--out BBSeq_MSA/VarList.OLINK_BBSeq_MSA_chr${CHNUM}
done

In [ ]:
%%bash
cd PCA_EDTA/subset_genetic_data

wc -l DementiaSeq1/VarList.OLINK_DementiaSeq1_chr*.snplist
wc -l DementiaSeq2/VarList.OLINK_DementiaSeq2_chr*.snplist
wc -l BBSeq_MSA/VarList.OLINK_BBSeq_MSA_chr*.snplist

cat DementiaSeq1/VarList.OLINK_DementiaSeq1_chr*.snplist DementiaSeq2/VarList.OLINK_DementiaSeq2_chr*.snplist BBSeq_MSA/VarList.OLINK_BBSeq_MSA_chr*.snplist NDRU-chip-genotyped/VarList.OLINK_NDRU-chip-genotyped.nodups.snplist | sort | uniq -c > VarList.allCohorts.counts.txt 

In [ ]:
%%bash
cd PCA_EDTA/subset_genetic_data

## make list of variants that are present across all cohorts
awk '{if($1 == 4) print}' VarList.allCohorts.counts.txt | awk '{print $2}' > VarList.allCohorts.shared.txt
wc -l VarList.allCohorts.counts.txt
wc -l VarList.allCohorts.shared.txt
head VarList.allCohorts.shared.txt

##### Subset again to keep only shared variants

In [ ]:
%%bash
cd PCA_EDTA/subset_genetic_data

module load plink/3.6-alpha

# DementiaSeq1
for CHNUM in {1..22}
do
plink2 \
--pfile /data/ALS_50k/DementiaSeq_QC/FILTERED.neurod.june2019.exclTOPMED.noDuplicates.noSpanningDels.hwe1e-6.VariantFreqOutliersRemoved.hg38.chr${CHNUM} \
--keep /data/ALS_50k/OLINK/Results/Oct2023/PCA_EDTA/SampleList.OLINK_DementiaSeq1.FID_IID.txt \
--extract VarList.allCohorts.shared.txt \
--make-bed \
--out DementiaSeq1/OLINK_DementiaSeq1_chr${CHNUM}
done

# DementiaSeq2
for CHNUM in {1..22}
do
plink2 \
--pfile /data/ALS_50k/BrainBankSeq_May2021/LBD.FTD/subset.LBD_FTD_RESv2/CLEAN.LBD.FTD.controls.res2.may2021.noSpanningDels.chr${CHNUM} \
--keep /data/ALS_50k/OLINK/Results/Oct2023/PCA_EDTA/SampleList.OLINK_DementiaSeq2.FID_IID.txt \
--extract VarList.allCohorts.shared.txt \
--make-bed \
--out DementiaSeq2/OLINK_DementiaSeq2_chr${CHNUM}
done

# BBSeq_MSA
for CHNUM in {1..22}
do
plink2 \
--pfile /data/ALS_50k/BrainBankSeq_May2021/MSA/WGS/MSA.keepRelated/FILTERED.MSA.keepRelated.controls_chr${CHNUM} \
--keep /data/ALS_50k/OLINK/Results/Oct2023/PCA_EDTA/SampleList.OLINK_BBSeq_MSA.FID_IID.txt \
--extract VarList.allCohorts.shared.txt \
--make-bed \
--out BBSeq_MSA/OLINK_BBSeq_MSA_chr${CHNUM}
done

## NDRU-chip-genotyped
plink2 \
--bfile /data/ALS_50k/OLINK/Results/Oct2023/Neurochip_olink/plink_output/FILTERED.NDRU.OLINK.hg38.keepRelated.autosomes \
--extract VarList.allCohorts.shared.txt \
--keep /data/ALS_50k/OLINK/Results/Oct2023/PCA_EDTA/SampleList.OLINK_NDRU-chip-genotyped.FID_IID.txt \
--make-bed \
--out NDRU-chip-genotyped/OLINK_NDRU-chip-genotyped-final

In [ ]:
%%bash
cd PCA_EDTA/subset_genetic_data

# make merge list of plink files
## DementiaSeq1
for CHNUM in {2..22}
do
echo "DementiaSeq1/OLINK_DementiaSeq1_chr${CHNUM}" >> DementiaSeq1/mergeList.OLINK_DementiaSeq1.txt
done

## DementiaSeq2
for CHNUM in {2..22}
do
echo "DementiaSeq2/OLINK_DementiaSeq2_chr${CHNUM}" >> DementiaSeq2/mergeList.OLINK_DementiaSeq2.txt
done

## BBSeq_MSA
for CHNUM in {2..22}
do
echo "BBSeq_MSA/OLINK_BBSeq_MSA_chr${CHNUM}" >> BBSeq_MSA/mergeList.OLINK_BBSeq_MSA.txt
done


In [ ]:
%%bash
cd PCA_EDTA/subset_genetic_data

# merge plink files 
module load plink/1.9.0-beta4.4
## DementiaSeq1
plink \
--bfile DementiaSeq1/OLINK_DementiaSeq1_chr1 \
--merge-list DementiaSeq1/mergeList.OLINK_DementiaSeq1.txt \
--make-bed \
--out DementiaSeq1/OLINK_DementiaSeq1_allChr

## DementiaSeq2
plink \
--bfile DementiaSeq2/OLINK_DementiaSeq2_chr1 \
--merge-list DementiaSeq2/mergeList.OLINK_DementiaSeq2.txt \
--make-bed \
--out DementiaSeq2/OLINK_DementiaSeq2_allChr

## BBSeq_MSA
plink \
--bfile BBSeq_MSA/OLINK_BBSeq_MSA_chr1 \
--merge-list BBSeq_MSA/mergeList.OLINK_BBSeq_MSA.txt \
--make-bed \
--out BBSeq_MSA/OLINK_BBSeq_MSA_allChr

##### merge all cohorts with shared variants

In [ ]:
%%bash
cd PCA_EDTA/subset_genetic_data

module load plink/1.9.0-beta4.4

plink \
--bfile NDRU-chip-genotyped/OLINK_NDRU-chip-genotyped-final \
--bmerge DementiaSeq1/OLINK_DementiaSeq1_allChr \
--make-bed \
--out temp1

plink \
--bfile temp1 \
--bmerge DementiaSeq2/OLINK_DementiaSeq2_allChr \
--make-bed \
--out temp2

plink \
--bfile temp2 \
--bmerge BBSeq_MSA/OLINK_BBSeq_MSA_allChr \
--make-bed \
--out merged_OLINK_cohorts.european

rm temp*


#### Generate genetic PCs

note: these are all European samples

In [ ]:
%%bash
cd PCA_EDTA/subset_genetic_data

#module load plink/1.9.0-beta4.4
#module load R/4.3
#module load GCTA/1.94.1
module load flashpca/2.0

# Prune snps
module load plink/3.6-alpha

plink2 \
--bfile merged_OLINK_cohorts.european \
--remove-nosex \
--maf 0.01 \
--geno 0.01 \
--hwe 5e-6 \
--autosome \
--exclude range /data/ALS_50k/DementiaSeq.TopmedJointCalled.June2020/FTD/Analysis.GLM/GLM.FTD.4Kcontrols/scripts/GenomicRangeExlusion.forPCA.txt \
--make-bed \
--out temp \
--memory 119500 --threads 19

plink2 \
--bfile temp \
--remove-nosex \
--geno 0.01 \
--maf 0.05 \
--indep-pairwise 1000 10 0.02 \
--out temp_pruning \
--memory 119500 --threads 19

plink2 \
--bfile temp \
--remove-nosex \
--extract temp_pruning.prune.in \
--keep-allele-order \
--make-bed \
--out pruned \
--memory 119500 --threads 19

# Calculate/generate PCs based on pruned data set
flashpca --bfile pruned --suffix _merged_OLINK_cohorts.european --numthreads 19

# Move all log files to a new folder
mkdir logFiles
mv *.log logFiles

# Remove intermediate files
rm temp*
rm pruned*

#### Update covariate file

In [ ]:
!mkdir PCA_EDTA_updated

In [ ]:
%%bash
cd PCA_EDTA_updated/
module load R/4.3
R --vanilla --no-save

# load libraries
require(data.table)
require(tidyverse)
library("Hmisc")
library(ggplot2) 
library(ggpubr)
library(ggrepel)
library(broom)
library(OlinkAnalyze)

# read in results files and pcs file
## note: had to redo this because noticed that two samples were merged incorrectly in the first iteration
## this happened because the two samples had NA in the Sample_ID column, etc
data <- fread("../OLINK_results_compiled_EDTA_updated.txt") %>% data.frame()
data$Age_Consensus <- ifelse(!is.na(data$Age_Onset), data$Age_Onset, data$Age_Collection)
data$Genome_ID[is.na(data$Genome_ID)] <- "NA"
data$NeuroChip_ID[is.na(data$NeuroChip_ID)] <- "NA"
table(data$Genome_ID)
table(data$NeuroChip_ID)

# without genetic data
temp_1 <- data %>% filter(Genome_ID == "NA" & NeuroChip_ID == "NA")
temp_1$IID <- temp_1$SampleID
table(temp_1$SampleID)
dim(temp_1)

# with genetic_data
temp_2 <- subset(data, !(data$SampleID %in% temp_1$SampleID))
temp_2_chip <- subset(temp_2, temp_2$NeuroChip_ID != "NA")
temp_2_chip$IID <- temp_2_chip$SampleID
table(temp_2_chip$SampleID)
dim(temp_2_chip)

temp_2_wgs <- subset(temp_2, !(temp_2$SampleID %in% temp_2_chip$SampleID))
temp_2_wgs$IID <- temp_2_wgs$Genome_ID
table(temp_2_wgs$SampleID)
table(temp_2_wgs$IID)
dim(temp_2_wgs)

# merge final
data_final <- rbind(temp_1, temp_2_chip, temp_2_wgs)
dim(data)
dim(data_final)

# merge data results and PCs
pcs <- fread("../PCA_EDTA/subset_genetic_data/pcs_merged_OLINK_cohorts.european",sep="\t") %>% select(-FID)

cov_no_missing_genetic <- merge(data_final, pcs, by="IID")
cov_with_missing_genetic <- subset(data_final, !(data_final$IID %in% cov_no_missing_genetic$IID))
cov_with_missing_genetic$PC1 <- NA
cov_with_missing_genetic$PC2 <- NA
cov_with_missing_genetic$PC3 <- NA
cov_with_missing_genetic$PC4 <- NA
cov_with_missing_genetic$PC5 <- NA
cov_with_missing_genetic$PC6 <- NA
cov_with_missing_genetic$PC7 <- NA
cov_with_missing_genetic$PC8 <- NA
cov_with_missing_genetic$PC9 <- NA
cov_with_missing_genetic$PC10 <- NA
cov_with_missing_genetic <- cov_with_missing_genetic %>% select(all_of(colnames(cov_no_missing_genetic)))
cov <- rbind(cov_no_missing_genetic, cov_with_missing_genetic)
cov$match_id <- ifelse(cov$SampleID == cov$Sample_ID, "yes","no")
cov$match_id_IID <- ifelse(cov$IID == cov$SampleID | cov$IID == cov$Genome_ID, "yes","no")
table(cov$match_id)
table(cov$match_id_IID)

dim(data)
dim(data_final)
dim(pcs)

# make sure that the updated file still have all the same samples
## first read in older file --> give it an index number for each sample
## then arrange the updated olink+pc file according to the same order of samples in the older file; based on SampleID
cov_old <- fread("../OLINK_results_compiled_EDTA_updated_wPCs.txt")
cov_old$Genome_ID[is.na(cov_old$Genome_ID)] <- "NA"
cov_old$NeuroChip_ID[is.na(cov_old$NeuroChip_ID)] <- "NA"
cov_old$match_id <- ifelse(cov_old$SampleID == cov_old$Sample_ID, "yes","no")
cov_old$match_id_IID <- ifelse(cov_old$IID == cov_old$SampleID | cov_old$IID == cov_old$Genome_ID, "yes","no")
table(cov_old$match_id)
table(cov_old$match_id_IID)
# check how many sample in the old olink+pc that were merged incorrectly --> from below,there were 12 samples! OMG!
cov_old %>% filter(match_id_IID == "no") %>% select(IID,Genome_ID,NeuroChip_ID,SampleID, Sample_ID, Diagnosis) %>% distinct()

cov_old_map <- cov_old %>% select(SampleID) %>% distinct()
cov_old_map$index <- seq(1:length(cov_old_map$SampleID))
tmp_complete <- merge(cov, cov_old_map, by="SampleID")
tmp_missing <- subset(cov, !(cov$SampleID %in% tmp_complete$SampleID)) # so there are no missing samples
dim(tmp_complete)
dim(tmp_missing)

cov_updated <- rbind(tmp_complete, tmp_missing) %>% arrange(index)
write.table(cov_updated, "../OLINK_results_compiled_EDTA_updated_wPCs_updatedNOV2024_FINAL.txt", sep="\t", quote=F, row.names=F, col.names=T)

# run step() to determine which PCs to correct for for the case/control status
## ALS vs controls
temp <- cov_updated %>%
        filter(Diagnosis == "ALS" | Diagnosis == "Control") %>%
        filter(!is.na(PC1)) %>%
        select(SampleID,Diagnosis,Sex,Age_Onset,Age_Consensus,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10) %>%
        distinct()
temp$CASE <- ifelse(temp$Diagnosis == "ALS", 1, 0)
temp$GENDER <- ifelse(temp$Sex == "Female", 1, 0)
table(temp$Diagnosis, temp$GENDER)
table(temp$CASE, temp$Sex)

# model using glm() and then use step() to determine the model that has the lowest AIC
# this tell us which covariates you included in the model are important to use/for adjustment in your analysis
model <- glm(CASE ~ GENDER + PC1 + PC2 + PC3 + PC4 + PC5 + PC6 + PC7 + PC8 + PC9 + PC10 , data = temp, family = "binomial"(link = "logit"))
summary(model)
step(model)

# run step() to determine which PCs to correct for for the AAO for cases only
model <- glm(Age_Onset ~ GENDER + PC1 + PC2 + PC3 + PC4 + PC5 + PC6 + PC7 + PC8 + PC9 + PC10 , data = temp %>% filter(Diagnosis=="ALS"), family = "gaussian")
summary(model)
step(model)

plot <- ggplot(temp, aes(x=PC1,y=PC2,color=Diagnosis)) +
        geom_point()
ggsave("PCplot_ALS_control.jpeg")

## Prep OLINK files for analysis

### Heparin

In [ ]:
!mkdir Heparin
!mkdir Heparin/Benchmarking
!mkdir Heparin/TestCohorts

In [ ]:
%%bash
cd Heparin/TestCohorts/
module load R/4.3
R --vanilla --no-save

require(data.table)
require(tidyverse)
library("Hmisc")
library(ggplot2) 
library(ggpubr)
library(ggrepel)
library(broom)
library(OlinkAnalyze)

# read in results files
data <- fread("../../OLINK_results_compiled_HEPARIN_wPCs.txt")
## regroup panels so that it is not split to I and II
data$Panel <- gsub("Cardiometabolic_II","Cardiometabolic",data$Panel)
data$Panel <- gsub("Inflammation_II","Inflammation",data$Panel)
data$Panel <- gsub("Neurology_II","Neurology",data$Panel)
data$Panel <- gsub("Oncology_II","Oncology",data$Panel)
data$SampleID_temp <- paste(data$SampleID,data$Olink_Plate_No,sep="_")
write.table(data, "../OLINK_results_compiled_HEPARIN_wPCs_tidy.txt",sep="\t",quote=F,row.names=F,col.names=T)

# sanity check to see how many duplicate samples are in the heparin plates - do only for ALS/Control samples
temp <- data %>% filter(Diagnosis == "ALS" | Diagnosis == "Control") %>% select(SampleID,Olink_Plate_No) %>% distinct()
temp$SampleID_new <- gsub("_1","",temp$SampleID)
temp_counts <- temp %>% select(SampleID_new,Olink_Plate_No) %>% distinct() %>% group_by(SampleID_new) %>% count() %>% arrange(n)
dups <- temp_counts %>% filter(n>1) %>% data.frame()
dups_samples <- subset(temp, temp$SampleID_new %in% dups$SampleID_new) %>% arrange(SampleID_new,SampleID)
dups_samples$SampleID_temp <- paste(dups_samples$SampleID,dups_samples$Olink_Plate_No,sep="_")
dups_samples
## note that there is one sample ITTO-RB46F-12-048C which were plated three times in plates 7,8 and 9
## to keep things simple, only keep dup pair samples on plate 9 and exclude the other dup pair sample on other plate
## will use the dup pairs as an internal check for inter-plate variablity check
dups_samples_toexclude <- dups_samples %>% filter(Olink_Plate_No != "TRAYNOR_9") 
dups_samples_toexclude
write.table(dups_samples_toexclude,"../SampleList.Heparin_ALS-controls_duplicates_to_exclude.txt",sep="\t",quote=F,row.names=F,col.names=T)

# run some QC; drop duplicate samples
data_nodups <- subset(data, !(data$SampleID %in% dups_samples_toexclude$SampleID))
samples_data_nodups <- data_nodups %>% select(SampleID,Diagnosis) %>% distinct()
dim(data_nodups)
table(samples_data_nodups$Diagnosis)
## check to see if there are assay/samples with QC_Warning; should be none
data_nodups %>% filter(QC_Warning == "WARN") %>% select(SampleID, Olink_Plate_No) %>% distinct()
## remove rows with missing NPX values
data_nodups_nomissNPX <- data_nodups %>% filter(!is.na(NPX))
dim(data_nodups_nomissNPX)
samples_data_nodups_nomissNPX <- data_nodups_nomissNPX %>% select(SampleID,Diagnosis) %>% distinct()
table(samples_data_nodups_nomissNPX$Diagnosis)

# plot scatterplot to visualise possible sample outliers
scatterplot_qc <- data_nodups_nomissNPX %>% 
                  mutate(SampleID_temp = SampleID) %>%
                  olink_qc_plot(color_g = "QC_Warning")
                  
ggsave("scatterplot_qc.jpeg", scatterplot_qc, width = 12, height = 13)


stats_persample <- data_nodups_nomissNPX %>%
                   group_by(SampleID_temp,Panel) %>%
                   summarise(Diagnosis = Diagnosis,
                             Panel = Panel,
                             median = median(NPX, na.rm = TRUE),
                              IQR = IQR(NPX, na.rm = TRUE))

mean_sample_median <- mean(stats_persample$median)
sd_sample_median <- sd(stats_persample$median)
mean_sample_IQR <- mean(stats_persample$IQR)
sd_sample_IQR <- sd(stats_persample$IQR)

lower_bound_median <- mean_sample_median - 3*sd_sample_median
upper_bound_median <- mean_sample_median + 3*sd_sample_median
lower_bound_IQR <- mean_sample_IQR - 3*sd_sample_IQR
upper_bound_IQR <- mean_sample_IQR + 3*sd_sample_IQR
                   
stats_persample$outlier <- ifelse(stats_persample$median > lower_bound_median &  stats_persample$median < upper_bound_median &
                                  stats_persample$IQR > lower_bound_IQR & stats_persample$IQR < upper_bound_IQR, "no", "yes")
samples_outliers <- stats_persample %>% select(SampleID_temp,Diagnosis,Panel,outlier) %>% distinct()

In [ ]:
from IPython.display import display
from PIL import Image

results="Heparin/TestCohorts/scatterplot_qc.jpeg"
display(Image.open(results))

#### Bridge normalization using overlap samples

In [ ]:
%%bash
cd Heparin/TestCohorts/
module load R/4.3
R --vanilla --no-save

require(data.table)
require(tidyverse)
library("Hmisc")
library(ggplot2) 
library(ggpubr)
library(ggrepel)
library(broom)
library(OlinkAnalyze)

# read in results files
data <- fread("../OLINK_results_compiled_HEPARIN_wPCs_tidy.txt")
#data$SampleID_new <- gsub("_1","",data$SampleID)

# prep data for bridging normalization across plates using the OlinkAnalyze package
# this is important to account for any potential batch effects as the plates were not all processed at the same time
## first, split to individual plates after removing one of the samples from the dup pairs
## i.e. only keep dup pair from samples from plate 9
temp <- split(data %>% select(-`ID.cryovial...1`,-`ID.cryovial...21`), data$Olink_Plate_No)
p5 <- temp[[1]] %>% as_tibble()
p6 <- temp[[2]] %>% as_tibble()
p7 <- temp[[3]] %>% as_tibble()
p8 <- temp[[4]] %>% as_tibble()
p9 <- temp[[5]] %>% as_tibble()

## Find overlapping samples - setting plate5 (i.e. TRAYNOR_5) as the reference plate 
## the overlapping samples across plates are the NIA_sample_control i.e. SC_1 and SC_2
overlap_samples <- intersect(p5$SampleID, p6$SampleID) %>% 
  data.frame() %>% 
  filter(!str_detect(., 'CONTROL_SAMPLE')) %>% #Remove internal OLINK control samples, if present
  pull(.)

## Perform Bridging normalization using the two overlapping samples across plates with the TRAYNOR_5 as the reference plate
## this step will adjust the NPX values for each plate using the Adj_factor calculated from the `olink_normalization` function
p5_p6 <- olink_normalization(df1 = p5, 
                             df2 = p6,
                             overlapping_samples_df1 = overlap_samples,
                             df1_project_nr = 'p5',
                             df2_project_nr = 'p6',
                             reference_project = 'p5')

p5_p7 <- olink_normalization(df1 = p5, 
                             df2 = p7,
                             overlapping_samples_df1 = overlap_samples,
                             df1_project_nr = 'p5',
                             df2_project_nr = 'p7',
                             reference_project = 'p5') %>%
        filter(Project == "p7")


p5_p8 <- olink_normalization(df1 = p5, 
                             df2 = p8,
                             overlapping_samples_df1 = overlap_samples,
                             df1_project_nr = 'p5',
                             df2_project_nr = 'p8',
                             reference_project = 'p5') %>%
        filter(Project == "p8")

p5_p9 <- olink_normalization(df1 = p5, 
                             df2 = p9,
                             overlapping_samples_df1 = overlap_samples,
                             df1_project_nr = 'p5',
                             df2_project_nr = 'p9',
                             reference_project = 'p5') %>%
        filter(Project == "p9")

## merge the NPX adjusted dataframes for each plate (i.e. 6-9) but NPX will not be adjusted for plate5 (because this is the refernce plate)
## --> make final dataframe without duplicates
temp_merge_norm <- rbind(p5_p6,p5_p7,p5_p8,p5_p9) %>% filter(!is.na(NPX)) %>% filter(QC_Warning == "PASS") %>% data.frame()
write.table(temp_merge_norm, "../OLINK_results_compiled_HEPARIN_wPCs_tidy_w-dups_post-bridged-normalization.txt",sep="\t",quote=F,row.names=F,col.names=T)

## run scatterplot qc to identify potential outliers; will use this to guide the decision on which dup to remove
scatterplot_qc <- temp_merge_norm %>% 
                  mutate(SampleID = SampleID_temp) %>%
                  olink_qc_plot(color_g = "Diagnosis")
ggsave("scatterplot_qc_post-bridged_normalization.jpeg", scatterplot_qc, width = 16, height = 16)

stats_persample <- temp_merge_norm %>%
                   group_by(SampleID_temp,Panel) %>%
                   summarise(Diagnosis = Diagnosis,
                             Panel = Panel,
                             median = median(NPX, na.rm = TRUE),
                              IQR = IQR(NPX, na.rm = TRUE))

mean_sample_median <- mean(stats_persample$median)
sd_sample_median <- sd(stats_persample$median)
mean_sample_IQR <- mean(stats_persample$IQR)
sd_sample_IQR <- sd(stats_persample$IQR)

lower_bound_median <- mean_sample_median - 3*sd_sample_median
upper_bound_median <- mean_sample_median + 3*sd_sample_median
lower_bound_IQR <- mean_sample_IQR - 3*sd_sample_IQR
upper_bound_IQR <- mean_sample_IQR + 3*sd_sample_IQR
                   
stats_persample$outlier <- ifelse(stats_persample$median >= lower_bound_median &  stats_persample$median <= upper_bound_median &
                                  stats_persample$IQR >= lower_bound_IQR & stats_persample$IQR <= upper_bound_IQR, "no", "yes")
samples_outliers <- stats_persample %>%
                    select(SampleID_temp,Diagnosis,Panel,outlier) %>%
                    distinct() %>%
                    filter(outlier == "yes")
samples_outliers %>% data.frame()
samples_outliers %>% data.frame() %>% select(SampleID_temp,Diagnosis) %>% distinct() %>% group_by(Diagnosis) %>% count()

## sanity check to see how many duplicate samples are in the heparin plates - do only for ALS/Control samples
## then make list dup samples to exclude after normalization
## if one of the dups is flagged an a potential outlier in plate 9, then remove that,
temp <- temp_merge_norm %>%
        filter(Diagnosis == "ALS" | Diagnosis == "Control") %>%
        select(SampleID,Olink_Plate_No) %>%
        distinct()
temp$SampleID_new <- gsub("_1","",temp$SampleID)
temp_counts <- temp %>%
               select(SampleID_new,Olink_Plate_No) %>%
               distinct() %>%
               group_by(SampleID_new) %>%
               count() %>%
               arrange(n)
dups <- temp_counts %>% filter(n>1) %>% data.frame()
dups_samples <- subset(temp, temp$SampleID_new %in% dups$SampleID_new) %>% arrange(SampleID_new,SampleID)
dups_samples$SampleID_temp <- paste(dups_samples$SampleID,dups_samples$Olink_Plate_No,sep="_")
dups_samples
## note that there is one sample ITTO-RB46F-12-048C which were plated three times in plates 7,8 and 9
## to keep things simple, only keep dup pair samples on plate 9 and exclude the other dup pair sample on other plate
## will use the dup pairs as an internal check for inter-plate variablity check
## else keep the dup on plate 9 and remove the other pair from the other plate
dups_samples$outlier <- ifelse(dups_samples$SampleID_temp %in% samples_outliers$SampleID_temp, "yes", "no")

dups_samples_toexclude_allplates <- dups_samples %>%
                                       group_by(SampleID_new) %>%
                                       mutate(to_exclude = ifelse(outlier == "yes", "yes", "no")) %>%
                                       filter(to_exclude == "yes") %>%
                                       arrange(SampleID_new) %>%
                                       data.frame()

tmp1 <- subset(dups_samples, !(dups_samples$SampleID_new %in% dups_samples_toexclude_allplates$SampleID_new)) %>%
        group_by(SampleID_new) %>%
        mutate(to_exclude = ifelse(Olink_Plate_No != "TRAYNOR_9", "yes", "no")) %>%
        arrange(SampleID_new) %>%
        data.frame()
    
tmp2 <- subset(dups_samples, dups_samples$SampleID_new %in% dups_samples_toexclude_allplates$SampleID_new) %>%
        group_by(SampleID_new) %>%
        filter(outlier == "no") %>%
        mutate(to_exclude = "no") %>%
        data.frame()

dups_samples_toexclude_final <- rbind(tmp1,tmp2,dups_samples_toexclude_allplates) %>% arrange(SampleID_new) %>% data.frame()
dups_samples_toexclude_final %>% data.frame()
dups_samples_toexclude_final_list <- dups_samples_toexclude_final %>% filter(to_exclude == "yes") %>% data.frame()
dups_samples_toexclude_final_list
write.table(dups_samples_toexclude_final,"../SampleList.Heparin_ALS-controls_duplicates_to_exclude_post-bridged-normalization.txt",sep="\t",quote=F,row.names=F,col.names=T)

## remove dups from temp_merge_norm dataframe --> save out clean file
merge_norm <- subset(temp_merge_norm, !(temp_merge_norm$SampleID_temp %in% dups_samples_toexclude_final_list$SampleID_temp))
merge_norm$possible_outlier <- ifelse(merge_norm$SampleID_temp %in% samples_outliers$SampleID_temp, "yes", "no")
write.table(merge_norm, "../OLINK_results_compiled_HEPARIN_wPCs_tidy_clean_post-bridged-normalization.txt",sep="\t",quote=F,row.names=F,col.names=T)

dim(temp_merge_norm)
dim(merge_norm)
#merge_norm$Diagnosis <- as.factor(merge_norm$Diagnosis)
#merge_norm$Sex <- as.factor(merge_norm$Sex)
#merge_norm$NPX_linear <- 2**merge_norm$NPX

## sanity check to make sure that the number of samples after removing one of the dup pair sample is correct
data %>% mutate(SampleID_new = gsub("_1","",SampleID)) %>% filter(Diagnosis == "ALS" | Diagnosis == "Control") %>% select(SampleID_new) %>% distinct() %>% count()
merge_norm %>% mutate(SampleID_new = gsub("_1","",SampleID)) %>% select(SampleID_new) %>% distinct() %>% count()
dups_samples_toexclude_final %>% select(SampleID_new) %>% distinct() %>% count()

## make sure that the samples that are not in merge_norm dataframe are the dups that were removed
temp1 <- data %>% mutate(SampleID_new = gsub("_1","",SampleID)) %>% filter(Diagnosis == "ALS" | Diagnosis == "Control") %>% select(SampleID_new,Diagnosis) %>% distinct()
temp2 <- merge_norm %>% mutate(SampleID_new = gsub("_1","",SampleID)) %>% select(SampleID_new,Diagnosis,possible_outlier) %>% distinct()
not_in_temp2 <- subset(temp1, !(temp1$SampleID_new %in% temp2$SampleID_new))
not_in_temp2$is_dup <- ifelse(not_in_temp2$SampleID_new %in% dups_samples_toexclude_final_list$SampleID_new, "yes","no")
not_in_temp2

## check number of samples by diagnosis
table(temp1$Diagnosis)
table(temp2$Diagnosis)
table(temp2$Diagnosis, temp2$possible_outlier)
table(not_in_temp2$Diagnosis)

# create PCA plot to visualise potential outliers
library(gridExtra)
pca_plot <- merge_norm %>%
            mutate(SampleID = SampleID_temp) %>%
            olink_pca_plot(color_g = "Diagnosis", byPanel = TRUE, outlierDefX = 6, outlierDefY = 6)
plot <- grid.arrange(pca_plot$Cardiometabolic, pca_plot$Inflammation, pca_plot$Neurology, pca_plot$Oncology, nrow = 2)
ggsave("pcaplot_post-bridged_normalization.jpeg", plot, device="jpeg", width = 16, height = 16)


In [ ]:
from IPython.display import display
from PIL import Image

display(Image.open("Heparin/TestCohorts/scatterplot_qc_post-bridged_normalization.jpeg"))
display(Image.open("Heparin/TestCohorts/pcaplot_post-bridged_normalization.jpeg"))

### EDTA
#### Bridge normalization using overlap samples

In [ ]:
!mkdir EDTA_updated
!mkdir EDTA_updated/Benchmarking
!mkdir EDTA_updated/TestCohorts

In [ ]:
%%bash
cd EDTA_updated/TestCohorts/
module load R/4.3.2
R --vanilla --no-save

require(data.table)
require(tidyverse)
library("Hmisc")
library(ggplot2) 
library(ggpubr)
library(ggrepel)
library(broom)
library(OlinkAnalyze)

# read in results files
data <- fread("../../OLINK_results_compiled_EDTA_updated_wPCs_updatedNOV2024_FINAL.txt") 
data$Panel <- gsub("Cardiometabolic_II","Cardiometabolic",data$Panel)
data$Panel <- gsub("Inflammation_II","Inflammation",data$Panel)
data$Panel <- gsub("Neurology_II","Neurology",data$Panel)
data$Panel <- gsub("Oncology_II","Oncology",data$Panel)
data$SampleID_temp <- paste(data$SampleID,data$Olink_Plate_No,sep="_")
#data$Age_Consensus <- ifelse(!is.na(data$Age_Onset), data$Age_Onset, data$Age_Collection)
#data$SampleID_new <- gsub("_1","",data$SampleID)

# prep data for bridging normalization across plates using the OlinkAnalyze package
# this is important to account for any potential batch effects as the plates were not all processed at the same time
## first, split to individual plates after removing one of the samples from the dup pairs
## i.e. only keep dup pair from samples from plate 9
temp <- split(data, data$Olink_Plate_No)
p1 <- temp[[1]] %>% as_tibble()
p2 <- temp[[2]] %>% as_tibble()
p3 <- temp[[3]] %>% as_tibble()
p4 <- temp[[4]] %>% as_tibble()


## Find overlapping samples - setting plate5 (i.e. TRAYNOR_5) as the reference plate 
## the overlapping samples across plates are the NIA_sample_control i.e. SC_1 and SC_2
overlap_samples <- intersect(p1$SampleID, p2$SampleID) %>% 
  data.frame() %>% 
  filter(!str_detect(., 'CONTROL_SAMPLE')) %>% #Remove internal OLINK control samples, if present
  pull(.)

## Perform Bridging normalization using the two overlapping samples across plates with the TRAYNOR_5 as the reference plate
## this step will adjust the NPX values for each plate using the Adj_factor calculated from the `olink_normalization` function
p1_p2 <- olink_normalization(df1 = p1, 
                             df2 = p2,
                             overlapping_samples_df1 = overlap_samples,
                             df1_project_nr = 'p1',
                             df2_project_nr = 'p2',
                             reference_project = 'p1')

p1_p3 <- olink_normalization(df1 = p1, 
                             df2 = p3,
                             overlapping_samples_df1 = overlap_samples,
                             df1_project_nr = 'p1',
                             df2_project_nr = 'p3',
                             reference_project = 'p1') %>%
        filter(Project == "p3")

p1_p4 <- olink_normalization(df1 = p1, 
                             df2 = p4,
                             overlapping_samples_df1 = overlap_samples,
                             df1_project_nr = 'p1',
                             df2_project_nr = 'p4',
                             reference_project = 'p1') %>%
        filter(Project == "p4")

## merge the NPX adjusted dataframes for each plate (i.e. 6-9) but NPX will not be adjusted for plate5 (because this is the refernce plate)
## --> make final dataframe without duplicates
temp_merge_norm <- rbind(p1_p2,p1_p3,p1_p4) %>% filter(!is.na(NPX)) %>% filter(QC_Warning == "PASS") %>% data.frame()
write.table(temp_merge_norm, "../OLINK_results_compiled_EDTA_w-dups_post-bridged-normalization.txt",sep="\t",quote=F,row.names=F,col.names=T)

## run scatterplot qc to identify potential outliers; will use this to guide the decision on which dup to remove
scatterplot_qc <- temp_merge_norm %>% 
                  mutate(SampleID = SampleID_temp) %>%
                  olink_qc_plot(color_g = "Diagnosis")
ggsave("scatterplot_qc_post-bridged_normalization.jpeg", scatterplot_qc, width = 16, height = 16)

stats_persample <- temp_merge_norm %>%
                   group_by(SampleID_temp,Panel) %>%
                   summarise(Diagnosis = Diagnosis,
                             Panel = Panel,
                             median = median(NPX, na.rm = TRUE),
                              IQR = IQR(NPX, na.rm = TRUE))

mean_sample_median <- mean(stats_persample$median)
sd_sample_median <- sd(stats_persample$median)
mean_sample_IQR <- mean(stats_persample$IQR)
sd_sample_IQR <- sd(stats_persample$IQR)

lower_bound_median <- mean_sample_median - 3*sd_sample_median
upper_bound_median <- mean_sample_median + 3*sd_sample_median
lower_bound_IQR <- mean_sample_IQR - 3*sd_sample_IQR
upper_bound_IQR <- mean_sample_IQR + 3*sd_sample_IQR
                   
stats_persample$outlier <- ifelse(stats_persample$median >= lower_bound_median &  stats_persample$median <= upper_bound_median &
                                  stats_persample$IQR >= lower_bound_IQR & stats_persample$IQR <= upper_bound_IQR, "no", "yes")
samples_outliers <- stats_persample %>%
                    select(SampleID_temp,Diagnosis,Panel,outlier) %>%
                    distinct() %>%
                    filter(outlier == "yes")
samples_outliers %>% data.frame()
samples_outliers %>% data.frame() %>% select(SampleID_temp,Diagnosis) %>% distinct() %>% group_by(Diagnosis) %>% count()

## sanity check to see how many duplicate samples are in the EDTA plates - do only for cases and control samples
## should not have any duplicate samples
temp <- temp_merge_norm %>%
        filter(Diagnosis != "NIA_sample_control" & Diagnosis != "Pooled_bringing_sample") %>%
        select(SampleID,Olink_Plate_No) %>%
        distinct()
temp$SampleID_new <- gsub("_1","",temp$SampleID)
temp_counts <- temp %>%
               select(SampleID_new,Olink_Plate_No) %>%
               distinct() %>%
               group_by(SampleID_new) %>%
               count() %>%
               arrange(n)
dups <- temp_counts %>% filter(n>1) %>% data.frame()
dups_samples <- subset(temp, temp$SampleID_new %in% dups$SampleID_new) %>% arrange(SampleID_new,SampleID)
dups_samples$SampleID_temp <- paste(dups_samples$SampleID,dups_samples$Olink_Plate_No,sep="_")
dups_samples

## remove dups (if any) from temp_merge_norm dataframe --> save out clean file
## since there are no dups, will just reassign new name to temp_merge_norm
merge_norm <- temp_merge_norm
merge_norm$possible_outlier <- ifelse(merge_norm$SampleID_temp %in% samples_outliers$SampleID_temp, "yes", "no")
write.table(merge_norm, "../OLINK_results_compiled_EDTA_wPCs_tidy_clean_post-bridged-normalization.txt",sep="\t",quote=F,row.names=F,col.names=T)

dim(temp_merge_norm)
dim(merge_norm)
#merge_norm$Diagnosis <- as.factor(merge_norm$Diagnosis)
#merge_norm$Sex <- as.factor(merge_norm$Sex)
#merge_norm$NPX_linear <- 2**merge_norm$NPX

## check number of samples by diagnosis
temp1 <- data %>% filter(Diagnosis != "NIA_sample_control" & Diagnosis != "Pooled_bringing_sample") %>% select(SampleID,Diagnosis) %>% distinct()
temp2 <- merge_norm %>% select(SampleID,Diagnosis,possible_outlier) %>% distinct()

table(temp1$Diagnosis)
table(temp2$Diagnosis)
table(temp2$Diagnosis, temp2$possible_outlier)


# create PCA plot to visualise potential outliers
library(gridExtra)
pca_plot <- merge_norm %>%
            mutate(SampleID = SampleID_temp) %>%
            olink_pca_plot(color_g = "Diagnosis", byPanel = TRUE, outlierDefX = 6, outlierDefY = 6)
plot <- grid.arrange(pca_plot$Cardiometabolic, pca_plot$Inflammation, pca_plot$Neurology, pca_plot$Oncology, nrow = 2)
ggsave("pcaplot_post-bridged_normalization.jpeg", plot, device="jpeg", width = 16, height = 16)

sessionInfo()

In [ ]:
from IPython.display import display
from PIL import Image

display(Image.open("EDTA_updated/TestCohorts/pcaplot_post-bridged_normalization.jpeg"))
display(Image.open("EDTA_updated/TestCohorts/scatterplot_qc_post-bridged_normalization.jpeg"))

## ***Updated 2024-11-13:*** Update phenotype info including plasma collection date for samples in EDTA cohort

1. Update dx for a handful of samples
2. Calculate ALSFRS pre-slopes

EDTA file to update: `/data/ALS_50k/OLINK/Results/Oct2023/EDTA_redo/OLINK_results_compiled_EDTA_wPCs_tidy_clean_post-bridged-normalization.txt`

ALSFRS file for HEPARIN cohrot (from Saro): `/data/ALS_50k/OLINK/Results/Oct2023/SampleList.forRuth_fromSaro_ALSFRS_preslope_June2024.csv`

Updated demographic/other info file from Justin Kwan: `/data/ALS_50k/OLINK/Results/Oct2023/Demographic_Data_for_13-N-0188_and_17-N-0131_Plasma_Samples_8-30-24_editRC.txt`

Note that 4 samples were labelled as ALS but are actually Asymptomatic ---  will also correct this here.

In [ ]:
%%bash
module load R/4.3.2
R --vanilla --no-save

require(data.table)
require(tidyverse)
library(lubridate) # this is needed to reformat dates for calculation of age

# read in old pheno file
old <- fread("/data/ALS_50k/OLINK/Results/Oct2023/EDTA_updated/OLINK_results_compiled_EDTA_wPCs_tidy_clean_post-bridged-normalization.txt")
old$Diagnosis[is.na(old$Diagnosis)] <- "NA"
dim(old)

temp <- old %>% select(SampleID,Diagnosis,PC1) %>% distinct()
temp_pc <- temp %>% filter(!is.na(PC1))
table(temp$Diagnosis)
table(temp_pc$Diagnosis)


# read in updated clinical file from justin -- note that there are 6 samples in which the diagnosis have been updated compared to what was in the old pheno file
updated_justin <- fread("/data/ALS_50k/OLINK/Results/Oct2023/Demographic_Data_for_13-N-0188_and_17-N-0131_Plasma_Samples_8-30-24_editRC.txt")
colnames(updated_justin) <- gsub(" ","_",colnames(updated_justin))
updated_justin$SampleID <- as.character(updated_justin$SampleID)
updated_justin <- updated_justin %>%
                  mutate(DOB_reformat = parse_date_time(DOB,orders = c("mdy", "dmy", "ymd"))) %>%
                  mutate(Date_Sample_collection_reformat = parse_date_time(Date_of_Sample_Collection,orders = c("mdy", "dmy", "ymd"))) %>%
                  mutate(Date_ALSFRS_reformat = parse_date_time(ALSFRS_Date,orders = c("mdy", "dmy", "ymd"))) %>%
                  mutate(Date_Onset_reformat = parse_date_time(Date_of_Sx_Onset,orders = c("mdy", "dmy", "ymd"))) %>%
                  mutate(Date_Current = Sys.Date())

updated_justin$Age_current <- interval(updated_justin$DOB_reformat,updated_justin$Date_Current) %>% as.numeric('years')
updated_justin$Age_onset_justin <- interval(updated_justin$DOB_reformat, updated_justin$Date_Onset_reformat) %>% as.numeric('years')
updated_justin$Age_collection_justin <- interval(updated_justin$DOB_reformat, updated_justin$Date_Sample_collection_reformat) %>% as.numeric('years')
updated_justin$YEARS_GAP_PLASMA_COLLECTED_TO_ONSET <- updated_justin$Age_collection_justin - updated_justin$Age_onset_justin
updated_justin$Age_ALSFRS <- interval(updated_justin$DOB_reformat, updated_justin$Date_ALSFRS_reformat) %>% as.numeric('years')
updated_justin$Interval_Sx_ALSFRS_months <- interval(updated_justin$Date_Onset_reformat,updated_justin$Date_ALSFRS_reformat)%/% months(1)
updated_justin$ALSFRS_preslope <- (48 - updated_justin$ALSFRS_Total_Score)/updated_justin$Interval_Sx_ALSFRS_months
write.table(updated_justin, "/data/ALS_50k/OLINK/Results/Oct2023/Demographic_Data_for_13-N-0188_and_17-N-0131_Plasma_Samples_8-30-24_editRC_wALSFRSscores.txt",sep="\t",quote=F,row.names=F,col.names=T)
table(updated_justin$Diagnosis, updated_justin$DX_short_justin)

# first, make subset of the samples in which information is to be updated
colsToKeep <- c("SampleID","DX_short_justin","DOB_reformat","Date_Sample_collection_reformat","Date_ALSFRS_reformat","Date_Onset_reformat",
                "ALSFRS_Total_Score","Age_onset_justin","Age_collection_justin","YEARS_GAP_PLASMA_COLLECTED_TO_ONSET", "Age_ALSFRS","Interval_Sx_ALSFRS_months","ALSFRS_preslope","Gender")
temp <- subset(updated_justin, updated_justin$SampleID %in% old$SampleID) %>% select(all_of(colsToKeep))
need_update <- merge(old, temp , by = "SampleID", all.y=T) 

no_update <- subset(old, !(old$SampleID %in% updated_justin$SampleID)) %>%
             mutate(DX_short_justin = "NA",
                    DOB_reformat = NA,
                    Date_Sample_collection_reformat = NA,
                    Date_ALSFRS_reformat = NA,
                    Date_Onset_reformat = NA,
                    ALSFRS_Total_Score = NA,
                    Age_onset_justin = NA,
                    Age_collection_justin = NA,
                    YEARS_GAP_PLASMA_COLLECTED_TO_ONSET = NA,
                    Age_ALSFRS = NA,
                    Interval_Sx_ALSFRS_months = NA,
                    ALSFRS_preslope = NA,
                    Gender = "NA") %>% select(any_of(colnames(need_update)))
need_update <- need_update %>% select(any_of(colnames(no_update)))
old_updated <- bind_rows(need_update, no_update)

# make final/updated diagnosis columns
old_updated$Sex_updated <- ifelse(is.na(old_updated$Sex), old_updated$Gender, old_updated$Sex)
old_updated$Diagnosis_updated <- ifelse(old_updated$DX_short_justin == "NA", old_updated$Diagnosis, old_updated$DX_short_justin)
old_updated$Age_Onset_updated <- ifelse(is.na(old_updated$Age_onset_justin), old_updated$Age_Onset, old_updated$Age_onset_justin)
old_updated$Age_Collection_updated <- ifelse(is.na(old_updated$Age_collection_justin), old_updated$Age_Collection, old_updated$Age_collection_justin)
old_updated$Age_Consensus_updated <- ifelse(is.na(old_updated$Age_Onset_updated), old_updated$Age_Collection_updated, old_updated$Age_Onset_updated)
old_updated$Age_Consensus_type <- ifelse(old_updated$Age_Onset_updated == old_updated$Age_Consensus_updated, "Age_Onset", "Age_Collection")
                   
old_updated_tidy <- old_updated %>%
                    mutate(Sex = Sex_updated,
                           Diagnosis = Diagnosis_updated,
                           Age_Onset = Age_Onset_updated,
                           Age_Collection = Age_Collection_updated,
                           Age_Consensus = Age_Consensus_updated) %>%
                    select(-Sex_updated,-Diagnosis_updated,-Age_Onset_updated,-Age_Collection_updated,-Age_Consensus_updated,-Age_onset_justin,-Age_collection_justin)
table(old_updated_tidy$Diagnosis, old_updated_tidy$Diagnosis_Details)

# sanity check to see how many samples were updated
old_pheno <- old %>% select(SampleID,Diagnosis_Status,Diagnosis,Diagnosis_Details,Sex) %>% distinct()
old_updated_pheno <- old_updated_tidy %>% select(SampleID,Diagnosis_Status,Diagnosis,Diagnosis_Details,Sex) %>% distinct()

table(old_pheno$Diagnosis,old_pheno$Sex)
table(old_updated_pheno$Diagnosis,old_updated_pheno$Sex)

# create new column to indicate grouped diagnosis for downstream analysis 
# i.e. neuro controls will be considered controls, and samples that are int he ALSFTD_sepctrum will be grouped together
old_updated_tidy$GroupedDiagnosis_temp1 <- old_updated_tidy$Diagnosis
old_updated_tidy$GroupedDiagnosis_temp1 <- ifelse(old_updated_tidy$GroupedDiagnosis_temp1 == "ALS-FTD" |
                                      old_updated_tidy$GroupedDiagnosis_temp1 == "FTD" |
                                      old_updated_tidy$GroupedDiagnosis_temp1 == "Asymptomatic",
                                      "ALSFTD_spectrum",NA)
old_updated_tidy$GroupedDiagnosis_temp2 <- old_updated_tidy$Diagnosis
old_updated_tidy$GroupedDiagnosis_temp2 <- ifelse(old_updated_tidy$GroupedDiagnosis_temp2 == "PD" |
                                      old_updated_tidy$GroupedDiagnosis_temp2 == "LBD" |
                                      old_updated_tidy$GroupedDiagnosis_temp2 == "MSA" |
                                      old_updated_tidy$GroupedDiagnosis_temp2 == "CBS" |
                                      old_updated_tidy$GroupedDiagnosis_temp2 == "PSP" |
                                      old_updated_tidy$GroupedDiagnosis_temp2 == "Dementia/Other",
                                      "Neuro_control",NA)
old_updated_tidy$GroupedDiagnosis_temp3 <- old_updated_tidy$Diagnosis
old_updated_tidy$GroupedDiagnosis_temp3 <- ifelse(old_updated_tidy$GroupedDiagnosis_temp3 == "Control",
                                      "Non-Neuro_control",NA)
old_updated_tidy$GroupedDiagnosis_temp4 <- old_updated_tidy$Diagnosis
old_updated_tidy$GroupedDiagnosis_temp4 <- ifelse(old_updated_tidy$GroupedDiagnosis_temp4 == "ALS",
                                      "ALS",NA)
                                      
old_updated_tidy$GroupedDiagnosis_temp5 <- ifelse(is.na(old_updated_tidy$GroupedDiagnosis_temp1), old_updated_tidy$GroupedDiagnosis_temp2, old_updated_tidy$GroupedDiagnosis_temp1)
old_updated_tidy$GroupedDiagnosis_temp6 <- ifelse(is.na(old_updated_tidy$GroupedDiagnosis_temp5), old_updated_tidy$GroupedDiagnosis_temp3, old_updated_tidy$GroupedDiagnosis_temp5)
old_updated_tidy$GroupedDiagnosis <- ifelse(is.na(old_updated_tidy$GroupedDiagnosis_temp6), old_updated_tidy$GroupedDiagnosis_temp4, old_updated_tidy$GroupedDiagnosis_temp6)
old_updated_tidy$Diagnosis_final <- ifelse(old_updated_tidy$GroupedDiagnosis == "ALS", "ALS", "Control")
old_updated_tidy <- old_updated_tidy %>% select(-GroupedDiagnosis_temp1,-GroupedDiagnosis_temp2,-GroupedDiagnosis_temp3,-GroupedDiagnosis_temp4,-GroupedDiagnosis_temp5,-GroupedDiagnosis_temp6)
write.table(old_updated_tidy,"/data/ALS_50k/OLINK/Results/Oct2023/EDTA_updated/OLINK_results_compiled_EDTA_wPCs_tidy_clean_post-bridged-normalization_updatedDx_wALSFRSscores_final.txt",sep="\t", quote=F,row.names=F,col.names=T)            

table(old_updated_tidy$Diagnosis, old_updated_tidy$GroupedDiagnosis)
table(old_updated_tidy$Diagnosis_final, old_updated_tidy$GroupedDiagnosis)
table(old_updated_tidy$Diagnosis, old_updated_tidy$Diagnosis_final)

temp <- old_updated_tidy %>% select(SampleID,Diagnosis,PC1) %>% distinct()
temp_pc <- temp %>% filter(!is.na(PC1))
temp_no_pc <- temp %>% filter(is.na(PC1))
table(temp$Diagnosis)
table(temp_pc$Diagnosis)
table(temp_no_pc$Diagnosis)

table(old_updated_tidy$GroupedDiagnosis)
table(old_updated_tidy$Diagnosis,old_updated_tidy$GroupedDiagnosis)

## ***Updated 2024-11-13:*** Update phenotype info including sample collection date for samples in HEPARIN cohort

will only update the ALSFRS preslope and sample collection date columns

File from Memoona with sample collection dates for the Heparin samples are saved on my computer: `/Users/chiarp/Desktop/collapse/NIH\ -\ BT/OLINK/Heparin_samples_plasma_collection_date.txt`. Made a copy here in this working directory.

In [ ]:
%%bash
module load R/4.3.2
R --vanilla --no-save

require(data.table)
require(tidyverse)

rm(list = ls())

# read in old pheno file
old <- fread("/data/ALS_50k/OLINK/Results/Oct2023/Heparin/OLINK_results_compiled_HEPARIN_wPCs_tidy_clean_post-bridged-normalization.txt")
old$Diagnosis[is.na(old$Diagnosis)] <- "NA"
dim(old)
table(old$Diagnosis)

# read in file from Saro
file_saro <- fread("/data/ALS_50k/OLINK/Results/Oct2023/SampleList.forRuth_fromSaro_ALSFRS_preslope_June2024.csv") %>%
             rename(ALSFRS_preslope = preslope,
                    ALSFRS_Total_Score = alsfrs_tot,
                    Age_Onset_updated = onset_age) %>%
             select(SampleID,ALSFRS_Total_Score,ALSFRS_preslope,Age_Onset_updated)

# create an alternative SampleID for Saro's file because there are a handful of samples in the OLINK data file that have "_1" as a suffix because it was run twice
file_saro$SampleID_alt <- paste(file_saro$SampleID,"_1",sep="")

need_update_1 <- merge(old, file_saro , by = "SampleID") %>% distinct() %>% select(-SampleID_alt)
tmp <- subset(old, !(old$SampleID %in% need_update_1$SampleID))
need_update_2 <- merge(tmp, file_saro , by.x = "SampleID", by.y="SampleID_alt") %>% distinct() %>% select(all_of(colnames(need_update_1)))

need_update <- rbind(need_update_1,need_update_2)
no_update <- subset(old, !(old$SampleID %in% need_update$SampleID)) %>%
             mutate(ALSFRS_preslope = NA,
                    ALSFRS_Total_Score = NA,
                    Age_Onset_updated = NA) %>% select(any_of(colnames(need_update))) %>% distinct()
need_update <- need_update %>% select(any_of(colnames(no_update)))
old_updated <- bind_rows(need_update, no_update)
table(old_updated$Diagnosis, old_updated$Diagnosis_Details)
dim(old)
dim(old_updated)
table(old_updated$Diagnosis)

colnames(old_updated)

# check to see how different the Age_Onset_updated is from the Age_Onset that is on file
temp <- old_updated %>% select(SampleID, Diagnosis, Age_Onset, Age_Onset_updated) %>% distinct() %>% filter(Diagnosis == "ALS")
temp$Age_Onset_diff <- temp$Age_Onset - temp$Age_Onset_updated

# check which samples have up updated onset age in saro's file
missing_update_saro <- temp %>% filter(is.na(Age_Onset_updated))
missing_update_saro

samples_with_diff_age <- temp %>% filter(Age_Onset_diff > 1 | Age_Onset_diff < -1)
samples_with_diff_age
# note there are 3 samples in which the Age_Onset on file is > 2 years different from Age_Onset_updated from Saro
# will use the Age_Onset_updated from Saro as the final onset age
# and for 4 samples where there is no onset age in Saro's file, we will use the info that is on file

# Update the Age_Onset info with Age_Onset_updated from Saro
old_updated$Age_Onset_onFile <- old_updated$Age_Onset
old_updated$Age_Onset <- ifelse(!is.na(old_updated$Age_Onset_updated), old_updated$Age_Onset_updated, old_updated$Age_Onset_onFile)
old_updated <- old_updated %>% mutate(YEARS_GAP_PLASMA_COLLECTED_TO_ONSET = Age_Collection - Age_Onset)

# read in file from Memoona with sample date collection date information she collected from the original tubes
file_dates <- fread("/data/ALS_50k/OLINK/Results/Oct2023/Heparin_samples_plasma_collection_date.txt",header=T) %>%
              select(-V4) %>% rename(Plasma_collected_date = Date, SampleID = `Sample name`) %>% select(-`Plate Name`) %>% distinct()
file_dates$SampleID <- gsub("ITTO-AD53F-13 166C","ITTO-AD53F-13_166C",file_dates$SampleID)
file_dates$Plasma_collected_date <- gsub("No date",NA,file_dates$Plasma_collected_date)
file_dates$Plasma_collected_date <- gsub(" ",NA,file_dates$Plasma_collected_date)
dim(file_dates)

# note that there are handful of samples in which the sample ID in Memoona's file need to be updated:
# ITTO-RM68F-12-043C = ITTO-RM68F-12-043C_1
# ITTO-BM44F-13-031 = ITTO-BM44F-13-031_1
# ITTO-RB46F-12-048C = ITTO-RB46F-12-048C_1
# ITTO-BC46F-11-0009 = ITTO-BC46F-11-0009_1
# ITTO-CI56F-11-0046 = ITTO-CI56F-11-0046_1
# ITTO-DI43F-11-0057 = ITTO-DI43F-11-0057_1
# ITTO-QR52F-11-0062 = ITTO-QR52F-11-0062_1
# ITTO-DF58M-12-047 = ITTO-DF58M-12-047_1
# ITTO-PF34F-13-002 = ITTO-PF34F-13-002_1
# ITTO-SM52F-12-071 = ITTO-SM52F-12-071_1
# ITTO-DA36M-12-069 = ITTO-DA36M-12-069_1
# ITTO-PA38M-12-060 = ITTO-PA38M-12-060_1
# ITTO-PV89F-13-052 = ITTO-PV89F-13-052_1
# ITTO-RM46M-13-132C = ITTO-RM46M-13-132C_1
# ITTO-SF71M-11-026C = ITTO-SF71M-11-026C_1
# ITTO-MA60M-12-065C = ITTO-MA60M-12-065C_1
# ITTO-ZM68F-12-071C = ITTO-ZM68F-12-071C_1
# ITTO-GD51F-12-094C = ITTO-GD51F-12-094C_1
# ITTO-CG46M-11-001C = ITTO-CG46M-11-001C_1
# ITTO-PM36F-11-022C = ITTO-PM36F-11-022C_1
sample_to_update_name <- c("ITTO-RM68F-12-043C","ITTO-BM44F-13-031","ITTO-RB46F-12-048C","ITTO-BC46F-11-0009","ITTO-CI56F-11-0046","ITTO-DI43F-11-0057","ITTO-QR52F-11-0062","ITTO-DF58M-12-047","ITTO-PF34F-13-002","ITTO-SM52F-12-071","ITTO-DA36M-12-069","ITTO-PA38M-12-060","ITTO-PV89F-13-052","ITTO-RM46M-13-132C","ITTO-SF71M-11-026C","ITTO-MA60M-12-065C","ITTO-ZM68F-12-071C","ITTO-GD51F-12-094C","ITTO-CG46M-11-001C","ITTO-PM36F-11-022C")
length(sample_to_update_name)
temp <- subset(file_dates, file_dates$SampleID %in% sample_to_update_name) %>% distinct()
temp$SampleID <- paste(temp$SampleID,"_1",sep="")

file_dates_updated <- rbind(file_dates,temp)

dim(temp)
dim(file_dates)
dim(file_dates_updated)

sample_only <- old_updated %>% select(SampleID) %>% distinct()

overlap <- subset(old_updated, old_updated$SampleID %in% file_dates_updated$SampleID)
no_overlap <- subset(old_updated, !(old_updated$SampleID %in% overlap$SampleID))

# sanity check to make sure all samples are accounted for
overlap_samples <- overlap %>% select(SampleID) %>% distinct()
no_overlap_samples <- no_overlap %>% select(SampleID) %>% distinct()
no_overlap_samples
dim(overlap_samples)
dim(no_overlap_samples)

# update the files
# note that there are five samples in which plasma was collected BEFORE onset by more than 1 year --> will flag these are asymptomatic at plasma collection date 
old_updated_wDates <- merge(old_updated, file_dates_updated, by="SampleID",all.x=T)
colnames(old_updated_wDates)

cases <- old_updated_wDates %>% filter(Diagnosis == "ALS")
cases$ALS_STATUS_AT_PLASMA_COLLECTION <- ifelse(cases$YEARS_GAP_PLASMA_COLLECTED_TO_ONSET >= -1.5, "Symptomatic", "Asymptomatic") # will the diff to 1.5 years because of the decimals

ctrls <- old_updated_wDates %>% filter(Diagnosis == "Control")
ctrls$ALS_STATUS_AT_PLASMA_COLLECTION <- "Control"

other <- old_updated_wDates %>% filter(!(SampleID %in% cases$SampleID) & !(SampleID %in% ctrls$SampleID))
other$ALS_STATUS_AT_PLASMA_COLLECTION <- NA

old_updated_wDates_final <- bind_rows(cases, ctrls,other) %>% arrange(Olink_Plate_No,Assay,Index)

old_updated_wDates_final$GroupedDiagnosis <- old_updated_wDates_final$ALS_STATUS_AT_PLASMA_COLLECTION
old_updated_wDates_final$GroupedDiagnosis <- gsub("Symptomatic", "ALS",old_updated_wDates_final$GroupedDiagnosis)
old_updated_wDates_final$GroupedDiagnosis <- gsub("Asymptomatic", "ALSFTD_spectrum",old_updated_wDates_final$GroupedDiagnosis)
old_updated_wDates_final$GroupedDiagnosis <- gsub("Control", "Non-Neuro_control",old_updated_wDates_final$GroupedDiagnosis)

old_updated_wDates_final$Diagnosis_final <- ifelse(old_updated_wDates_final$GroupedDiagnosis == "ALS", "ALS", "Control")
table(old_updated_wDates_final$Diagnosis,old_updated_wDates_final$Diagnosis_final)
table(old_updated_wDates_final$Diagnosis_final,old_updated_wDates_final$GroupedDiagnosis)

# check which are the two samples that are `Asymptomatic`
old_updated_wDates_final[,c(1,21:42,57:64)] %>% filter(YEARS_GAP_PLASMA_COLLECTED_TO_ONSET < -1.5) %>% distinct()

# note that ITTO-MC49F-14-040 is a phenoconverted sample. Plasma was collected -7.39 years before symptom onset
# will update the Diagnosis in the file to 'Asymptomatic'; the GroupedDiagnosis is "ALSFTD_spectrum"
old_updated_wDates_final[old_updated_wDates_final$SampleID == "ITTO-MC49F-14-040","Diagnosis"] <- "Asymptomatic"
table(old_updated_wDates_final$Diagnosis,old_updated_wDates_final$Diagnosis_final)
table(old_updated_wDates_final$Diagnosis_final,old_updated_wDates_final$GroupedDiagnosis)

# save out file
write.table(old_updated_wDates_final,"/data/ALS_50k/OLINK/Results/Oct2023/Heparin/OLINK_results_compiled_HEPARIN_wPCs_tidy_clean_post-bridged-normalization_wALSFRSscores_final.txt",sep="\t", quote=F,row.names=F,col.names=T)            